# Chapter 6 · Describing Two Variables: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, one month of the triage log of an emergency department, through the director's questions; only the sections with a calculation appear (6.2, 6.3, and 6.5 to 6.11). The book draws the figures; the notebook prints the same numbers. **Part B** gives the written solution of every chapter exercise (1 to 14, and the technical note on the coordinates of the perceptual maps; statements are in the book) and, for the exercises whose solution needs a calculation (2, 4, 7, 9, 11, 13, and 14), its code and result. The code is Python 3 with pandas. Run the cells in order.

## Part A. The case

**Data.** `data/triage_month.csv`: 1,263 rows, one per emergency visit in March 2026, and 13 columns: `visit_id`, `date`, `arrived_at`, `seen_at`, `arrival_route` (1 walk-in, 2 ambulance, 3 transfer), `sex`, `age_years`, `triage_level` (1 most urgent to 5 least urgent), `pain`, `temp_c`, `heart_rate`, `prior_visits`, and `admitted`. Each cell carries the number of the book section it belongs to.

### 6.2 The contingency table

*How does admission differ across the arrival routes?*

In [1]:
# 1. Read the March triage log: one row per visit
import pandas as pd

df = pd.read_csv("data/triage_month.csv")

# 2. The contingency table of arrival route and admission, with the totals (margins)
table = pd.crosstab(df["arrival_route"], df["admitted"], margins=True)
print(table)

# 3. Row profiles: the percentage of each row (arrival route) that falls in each column
rows = pd.crosstab(df["arrival_route"], df["admitted"], normalize="index")
print((100 * rows).round(1))

admitted        no  yes   All
arrival_route                
1              722  199   921
2              152   81   233
3               74   35   109
All            948  315  1263
admitted         no   yes
arrival_route            
1              78.4  21.6
2              65.2  34.8
3              67.9  32.1


### 6.3 Graphs for two qualitative variables

*How can the two variables be shown together?*

In [2]:
# 1. Counts of each arrival route by admission, admitted visits first (the book draws the grouped bars)
counts = pd.crosstab(df["arrival_route"], df["admitted"]).reindex(columns=["yes", "no"])
print(counts)

# 2. Row profiles in %, which scale every stacked bar to 100 (the book draws the stacked bars)
profiles = 100 * pd.crosstab(df["arrival_route"], df["admitted"], normalize="index")
print(profiles.reindex(columns=["yes", "no"]).round(1))

admitted       yes   no
arrival_route          
1              199  722
2               81  152
3               35   74
admitted        yes    no
arrival_route            
1              21.6  78.4
2              34.8  65.2
3              32.1  67.9


### 6.5 Phi and Cramér's V

*How strong is the association, on a scale from 0 to 1?*

In [3]:
# 1. Expected counts if arrival route and admission were independent, and the chi-square statistic (no continuity correction)
from scipy.stats import chi2_contingency

observed = pd.crosstab(df["arrival_route"], df["admitted"])
chi2, _, _, expected = chi2_contingency(observed, correction=False)
print("Expected counts (columns: discharged, admitted):")
print(expected.round(1))

# 2. Chi-square and Cramer's V for arrival route
n = len(df)
print("Arrival route: chi-square", round(chi2, 2), "; Cramer's V", round((chi2 / n) ** 0.5, 3))

# 3. The same for triage level and admission
levels = pd.crosstab(df["triage_level"], df["admitted"])
chi2_levels = chi2_contingency(levels, correction=False)[0]
print("Triage level: chi-square", round(chi2_levels, 1), "; Cramer's V", round((chi2_levels / n) ** 0.5, 3))

Expected counts (columns: discharged, admitted):
[[691.3 229.7]
 [174.9  58.1]
 [ 81.8  27.2]]
Arrival route: chi-square 20.47 ; Cramer's V 0.127
Triage level: chi-square 388.2 ; Cramer's V 0.554


### 6.6 Perceptual maps and correspondence analysis

*How can a table with many categories be drawn as a map?*

In [4]:
# 1. Row profiles of arrival route by triage level, and the share of each triage level in the whole log
route_level = pd.crosstab(df["arrival_route"], df["triage_level"])
row_profiles = route_level.div(route_level.sum(axis=1), axis=0)
column_share = route_level.sum(axis=0) / len(df)

# 2. Chi-square distance between the profiles of walk-in (1) and ambulance (2) visits, and the total inertia (chi-square divided by n)
difference = row_profiles.loc[1] - row_profiles.loc[2]
distance = ((difference ** 2 / column_share).sum()) ** 0.5
chi2_map = chi2_contingency(route_level, correction=False)[0]
print("Distance between the walk-in and ambulance profiles:", round(distance, 3), "; total inertia:", round(chi2_map / len(df), 5))

Distance between the walk-in and ambulance profiles: 0.871 ; total inertia: 0.11867


In [5]:
# 3. Correspondence analysis by hand: standardized residuals and their singular value decomposition
import numpy as np

p = route_level.to_numpy(dtype=float) / len(df)
r = p.sum(axis=1)
c = p.sum(axis=0)
expected_share = r[:, None] * c[None, :]
s = (p - expected_share) / np.sqrt(expected_share)
u, singular, vt = np.linalg.svd(s, full_matrices=False)
row_xy = u[:, :2] * singular[:2] / np.sqrt(r[:, None])
column_xy = vt.T[:, :2] * singular[:2] / np.sqrt(c[:, None])

# 4. The share of the inertia carried by each axis, and the distance between walk-in and ambulance on the map
axis_share = 100 * singular[:2] ** 2 / (singular ** 2).sum()
print("Inertia carried by the first and second axes (%):", axis_share.round(1).tolist())
print("Distance between walk-in and ambulance on the map:", round(np.sqrt(((row_xy[0] - row_xy[1]) ** 2).sum()), 3))

Inertia carried by the first and second axes (%): [97.4, 2.6]
Distance between walk-in and ambulance on the map: 0.871


### 6.7 A quantitative variable described by group

*How do age and heart rate differ between admitted and discharged visits?*

In [6]:
# 1. Age by admission status: count, mean, standard deviation, and median (the book draws the boxplots)
by_group = df.groupby("admitted")
print(by_group["age_years"].agg(["count", "mean", "std", "median"]).round(1))

# 2. Heart rate: the three quartiles in each group
print(by_group["heart_rate"].quantile([0.25, 0.5, 0.75]).round(1))

          count  mean   std  median
admitted                           
no          948  51.7  17.9    52.0
yes         315  55.2  19.1    55.0
admitted      
no        0.25     84.0
          0.50     93.0
          0.75    102.0
yes       0.25     87.5
          0.50     97.0
          0.75    107.5
Name: heart_rate, dtype: float64


### 6.8 The scatter plot

*How do heart rate and temperature move together?*

In [7]:
# 1. Heart rate and temperature in classes (the book draws the scatter plot): the count of each combination
hr_class = pd.cut(df["heart_rate"], [-np.inf, 80, 100, 120, np.inf], right=False)
temp_class = pd.cut(df["temp_c"], [-np.inf, 37, 38, 39, np.inf], right=False)
joint = pd.crosstab(hr_class, temp_class)
print("Counts by heart-rate class (rows) and temperature class (columns):", joint.to_numpy().tolist())

Counts by heart-rate class (rows) and temperature class (columns): [[123, 76, 0, 1], [321, 240, 33, 44], [133, 91, 77, 87], [3, 3, 10, 21]]


### 6.9 Covariance

*Do heart rate and temperature rise together, and by how much?*

In [8]:
# 1. Sample covariance of heart rate and temperature: the first eight visits (hand calculation) and the full month
first = df.head(8)
print("Covariance, first eight visits:", round(first["heart_rate"].cov(first["temp_c"]), 2))
print("Covariance, full month:", round(df["heart_rate"].cov(df["temp_c"]), 2))

Covariance, first eight visits: 4.61
Covariance, full month: 6.25


### 6.10 Pearson's correlation coefficient

*How strong is the linear association, free of the units?*

In [9]:
# 1. Pearson correlation of heart rate and temperature: the first eight visits and the full month
print("Correlation, first eight visits:", round(first["heart_rate"].corr(first["temp_c"]), 2))
print("Correlation, full month:", round(df["heart_rate"].corr(df["temp_c"]), 3))

# 2. The correlation matrix of age, heart rate, and temperature
print(df[["age_years", "heart_rate", "temp_c"]].corr().round(2))

Correlation, first eight visits: 0.18
Correlation, full month: 0.427
            age_years  heart_rate  temp_c
age_years        1.00       -0.00   -0.03
heart_rate      -0.00        1.00    0.43
temp_c          -0.03        0.43    1.00


### 6.11 Rank correlation for ordered variables

*How are two ordered variables related?*

In [10]:
# 1. Pain categories as codes in the order of the scale (only to rank them; the steps are not claimed to be equal)
pain_code = df["pain"].map({"none": 0, "mild": 1, "moderate": 2, "severe": 3})

# 2. Spearman correlation of triage level with pain: the first eight visits and the full month
print("Spearman, triage level and pain, first eight visits:", round(first["triage_level"].corr(pain_code.head(8), method="spearman"), 2))
print("Spearman, triage level and pain, full month:", round(df["triage_level"].corr(pain_code, method="spearman"), 2))

# 3. Spearman correlation of triage level with heart rate
print("Spearman, triage level and heart rate:", round(df["triage_level"].corr(df["heart_rate"], method="spearman"), 2))

Spearman, triage level and pain, first eight visits: -0.47
Spearman, triage level and pain, full month: -0.25
Spearman, triage level and heart rate: -0.27


## Part B. Exercise solutions

### 6.14 Exercise 1. Sex and admission

| Sex | Admitted | Discharged | Total |
|---|---|---|---|
| Female | 150 | 504 | 654 |
| Male | 165 | 444 | 609 |
| Total | 315 | 948 | 1,263 |



For the admitted men: row percentage $165/609=27.1\%$, the proportion of men
who were admitted; column percentage $165/315=52.4\%$, the proportion of
admitted patients who were men; joint percentage $165/1{,}263=13.1\%$, the
proportion of all visits that were admitted men. The row percentage answers
the question, compared with $150/654=22.9\%$ among women: men were admitted
somewhat more often.

### 6.14 Exercise 2. Pain and admission

The row profiles give admission in 16.7% of the 293 visits with no pain,
23.1% of the 432 with mild pain, 29.8% of the 372 with moderate pain, and
33.1% of the 166 with severe pain: the admitted proportion rises through the
pain scale.

In [11]:
# 1. Row profiles (%) of admission by pain, in the order of the scale
order = ["none", "mild", "moderate", "severe"]
pain = pd.Categorical(df["pain"], categories=order, ordered=True)
print((100 * pd.crosstab(pain, df["admitted"], normalize="index")).round(1))

admitted    no   yes
row_0               
none      83.3  16.7
mild      76.9  23.1
moderate  70.2  29.8
severe    66.9  33.1


### 6.14 Exercise 3. Expected counts, chi-square, and phi

Expected counts: female admitted $654\times315/1{,}263=163.1$, female
discharged $490.9$, male admitted $151.9$, male discharged $457.1$; the rows
add to 654 and 609 and the columns to 315 and 948. Every observed count differs
from its expectation by 13.1, so
$\chi^2=171.6\,(1/163.1+1/490.9+1/151.9+1/457.1)=1.05+0.35+1.13+0.38=2.91$
and $\phi=\sqrt{2.91/1{,}263}=0.048$. Against $0.127$ for arrival route, the
association of admission with sex is smaller still; both are weak beside the
$0.554$ for triage level.

### 6.14 Exercise 4. Cramér's V for pain and for triage level

Both tables have two columns, so $\min(r-1,c-1)=1$ and $V=\phi$ numerically;
$V$ uses a scale from 0 to 1, but the category proportions and their definitions
still affect its value. To verify the upper bound, write $R_i$ and $C_j$ for
the positive row and column totals. Expanding the squared differences gives

$$ \frac{\chi^2}{n}=\sum_{i,j}\frac{O_{ij}^2}{R_iC_j}-1. $$

Since $O_{ij}\leq C_j$, the sum is at most
$\sum_{i,j}O_{ij}/R_i=r$. Using $O_{ij}\leq R_i$ instead bounds it by $c$.
Thus $0\leq\chi^2/n\leq\min(r-1,c-1)$, and division by this upper bound
followed by a square root gives $0\leq V\leq1$.
Admission is associated with pain, $V=0.13$,
about as weakly as with arrival route, and with triage level far more
strongly, $V=0.55$.

In [12]:
# 1. Chi-square and Cramer's V of pain and of triage level against admission
n = len(df)
for col in ["pain", "triage_level"]:
    chi2 = chi2_contingency(pd.crosstab(df[col], df["admitted"]), correction=False)[0]
    print(col + ": chi-square", round(chi2, 1), "; Cramer's V", round((chi2 / n) ** 0.5, 3))

pain: chi-square 22.0 ; Cramer's V 0.132
triage_level: chi-square 388.2 ; Cramer's V 0.554


### 6.14 Exercise 5. Spearman by hand

Pain codes 3, 2, 2, 1, 0, 1 receive ranks 6, 4.5, 4.5, 2.5, 1, 2.5 (the two 2s
share ranks 4 and 5; the two 1s share 2 and 3). Triage levels 1, 2, 3, 3, 5, 4
receive ranks 1, 2, 3.5, 3.5, 6, 5. Pearson's coefficient of the two rank
series is $\rho=-0.93$: the more severe the pain, the more urgent (lower) the
level. The differences $d_i$ are 5, 2.5, 1, $-1$, $-5$, $-2.5$, with
$\sum d_i^2=64.5$, and the shortcut gives $1-6(64.5)/(6\times35)=-0.84$,
not $-0.93$; the shortcut assumes no ties and is wrong here, where both series
have them.

### 6.14 Exercise 6. Group differences and individual overlap

The means summarize groups, while the overlapping boxes show that many ages
occur in both. An older age is not a rule that determines admission. The
shorter wait among admitted visits is an association: triage level influences
both the waiting rule and admission in this synthetic example. Understanding
why the groups differ requires information about that process and relevant
patient characteristics; the two medians alone do not establish an effect
of admission on waiting time.

### 6.14 Exercise 7. Temperature and pain by admission status

Temperature: 37.47 (1.15) among the admitted, 37.25 (1.03) among the
discharged, in degrees Celsius. Histograms show a lower-temperature group and
a smaller febrile group; a single mean and SD conceal this structure. Pain: none 49 (15.6%) against 244 (25.7%), mild 100 (31.7%)
against 332 (35.0%), moderate 111 (35.2%) against 261 (27.5%), severe 55
(17.5%) against 111 (11.7%).

In [13]:
# 1. Temperature by admission status: mean and standard deviation
temp = df.groupby("admitted")["temp_c"].agg(["mean", "std"]).round(2)
print(temp)

# 2. Pain by admission status: counts and percentages of each column
counts = pd.crosstab(pain, df["admitted"])
share = (100 * pd.crosstab(pain, df["admitted"], normalize="columns")).round(1)
for level in order:
    print("Pain", level + ": admitted", counts.loc[level, "yes"], "(" + str(share.loc[level, "yes"]) + "%); discharged", counts.loc[level, "no"], "(" + str(share.loc[level, "no"]) + "%)")

           mean   std
admitted             
no        37.25  1.03
yes       37.47  1.15
Pain none: admitted 49 (15.6%); discharged 244 (25.7%)
Pain mild: admitted 100 (31.7%); discharged 332 (35.0%)
Pain moderate: admitted 111 (35.2%); discharged 261 (27.5%)
Pain severe: admitted 55 (17.5%); discharged 111 (11.7%)


### 6.14 Exercise 8. Covariance and correlation by hand

Means $\bar{x}=57.4$ years and $\bar{y}=55.0$ minutes. Deviations of age:
$-26.4$, 23.6, 13.6, 11.6, $-22.4$; of waiting time: 92, $-12$, $-12$, $-50$,
$-18$. Products: $-2{,}428.8$, $-283.2$, $-163.2$, $-580.0$, 403.2; sum
$-3{,}052$; covariance $-3{,}052/4=-763$ year-minutes. Standard deviations
$s_x=22.78$ years and $s_y=53.8$ minutes, so
$r=-763/(22.78\times53.8)=-0.62$, with no unit. In these five visits the older
patients waited less; five visits describe nothing beyond themselves.

### 6.14 Exercise 9. Correlation matrix and triage level against waiting time

The matrix shows little linear association of age with the other three, heart rate and temperature
at $0.43$, and waiting time weakly negatively related to heart rate ($-0.22$)
and temperature ($-0.14$). Spearman's coefficient between triage level and
waiting time is $0.82$: the less urgent the level, the longer the wait, almost
monotonically. It is large because the department's rule produces it: triage
level is the criterion by which patients are called, so the relation is built
into the process rather than discovered in the patients.

In [14]:
# 1. Waiting time in minutes, and the correlation matrix of age, heart rate, temperature, and waiting time
df["wait"] = (pd.to_datetime(df["seen_at"]) - pd.to_datetime(df["arrived_at"])).dt.total_seconds() / 60
print(df[["age_years", "heart_rate", "temp_c", "wait"]].corr().round(2))

# 2. Spearman correlation of triage level with waiting time
print("Spearman, triage level and waiting time:", round(df["triage_level"].corr(df["wait"], method="spearman"), 2))

            age_years  heart_rate  temp_c  wait
age_years        1.00       -0.00   -0.03 -0.00
heart_rate      -0.00        1.00    0.43 -0.22
temp_c          -0.03        0.43    1.00 -0.14
wait            -0.00       -0.22   -0.14  1.00
Spearman, triage level and waiting time: 0.82


### 6.14 Exercise 10. What the coefficient fails to show

(a) Agreement: two oximeters can have a systematic offset as well as measurement noise
and still give $r=0.97$; the differences between paired readings are needed
(Section 6.12, “Not agreement”). (b) A relation
between individuals: the correlation is between hospital averages and says
nothing about whether an older patient is more likely to be admitted (“Not a
statement about individuals”). (c) The strength of the relation in
general: intensive-care patients are a restricted range of lactate and of
risk, which can alter $r$ (“Not independent of the range studied”). (d)
Causation: the two measures rise and fall together because both reflect
glycemia; $r$ cannot say that one produces the other (“Not causation”).
(e) A causal effect: hospital size and activity can account for both larger
parking capacity and more deaths. The observed correlation gives no basis
for expecting fewer deaths after removing parking spaces (“Not causation”).
Mortality comparisons also require appropriate denominators and attention
to the patients treated; annual counts alone do not measure quality of care.

### 6.14 Exercise 11. Fahrenheit

The covariance is multiplied by 1.8, from 6.25 to 11.25, because every
temperature deviation is; the correlation is unchanged at 0.427. The
covariance shows that it carries the units of both variables; the correlation
illustrates part (a) of Proposition 6.2, invariance under a
positive rescaling and a shift.

In [15]:
# 1. Temperature in Fahrenheit, and the covariance and correlation of heart rate with each unit
temp_f = 1.8 * df["temp_c"] + 32
print("Covariance, Celsius:", round(df["heart_rate"].cov(df["temp_c"]), 2), "; Fahrenheit:", round(df["heart_rate"].cov(temp_f), 2))
print("Correlation, Celsius:", round(df["heart_rate"].corr(df["temp_c"]), 3), "; Fahrenheit:", round(df["heart_rate"].corr(temp_f), 3))

Covariance, Celsius: 6.25 ; Fahrenheit: 11.25
Correlation, Celsius: 0.427 ; Fahrenheit: 0.427


### 6.14 Exercise 12. Choosing the tool from the scales

(a) Both nominal: contingency table with row profiles, grouped or stacked
bars, $\phi$ or $V$. (b) Both ordinal: Spearman's $\rho$ on the ranks, since
the order is the information; a contingency table can display the counts. (c)
Nominal and quantitative: median [$Q_1$, $Q_3$] of length of stay by ward,
if the within-ward distributions are skewed, and side-by-side boxplots. (d) Both quantitative:
scatter plot, then Pearson's $r$, if the plot shows that a linear summary is appropriate. (e) A binary outcome against a twelve-level nominal
variable: a $12\times2$ contingency table with the readmission proportion of
each ward as the row profile, a bar chart of those proportions with each
ward's $n$, and Cramér's $V$ for the overall association; the profiles, not
the coefficient, are what the management report acts on.

### 6.14 Exercise 13. Checking a perceptual map

Levels 1 and 2 account for $27/109=24.8\%$ of transfers and
$86/921=9.3\%$ of walk-ins. For level 1, the contribution to their squared
distance is

$$ \frac{(4/109-12/921)^2}{44/1,263}\approx0.01608. $$

The following block recomputes both complete profiles and their distance
from the visit log. It can run independently of the chapter's coordinate code.

The plotted coordinates give
$\sqrt{(0.20679+0.19531)^2+(0.17624+0.01122)^2}\approx0.444$,
agreeing with the table calculation. The first axis retains 97.4% of the
inertia, not 97.4% of patient outcomes or correct predictions. The figure
describes this register; assessing sampling uncertainty requires a separate
inferential analysis.

In [16]:
# 1. Distance between the profiles of transfer (3) and walk-in (1) visits by triage level
route_level = pd.crosstab(df["arrival_route"], df["triage_level"])
profiles = route_level.div(route_level.sum(axis=1), axis=0)
column_share = route_level.sum(axis=0) / len(df)
difference = profiles.loc[3] - profiles.loc[1]
print("Distance between the transfer and walk-in profiles:", round(((difference ** 2 / column_share).sum()) ** 0.5, 3))

Distance between the transfer and walk-in profiles: 0.444


### 6.14 Exercise 14. Joint classes and their denominators

There is one visit with temperature at least 38.0 ° C among the
200 visits with heart rate below 80 beats/min. The other three classes have
77 of 638, 164 of 388, and 31 of 37. Their proportions are therefore
0.5%, 12.1%, 42.3%, and 83.8%. The following code reconstructs the
counts and the four conditional proportions.

The largest count of visits at or above the temperature cutpoint is in the
100-119 beats/min class; the largest within-class proportion is in the
120-or-more class. Counts and profiles answer different questions. The
scatter plot retains the actual paired measurements, including their
variation within each class. Grouping replaces each measurement by an
interval, and the apparent pattern can change with the chosen boundaries.

In [17]:
# 1. Joint classes of heart rate and temperature
hr_class = pd.cut(df["heart_rate"], [-np.inf, 80, 100, 120, np.inf], right=False)
temp_class = pd.cut(df["temp_c"], [-np.inf, 37, 38, 39, np.inf], right=False)
joint = pd.crosstab(hr_class, temp_class)
print("Counts:", joint.to_numpy().tolist())

# 2. The share of each heart-rate class with a temperature of 38 or more (%)
print("Share with temperature of 38 or more, by heart-rate class (%):", (100 * joint.iloc[:, 2:].sum(axis=1) / joint.sum(axis=1)).round(1).tolist())

Counts: [[123, 76, 0, 1], [321, 240, 33, 44], [133, 91, 77, 87], [3, 3, 10, 21]]
Share with temperature of 38 or more, by heart-rate class (%): [0.5, 12.1, 42.3, 83.8]


### 6.14 Technical note: coordinates of the perceptual maps

This note supplies the coordinate derivation used in
Section 6.6. A matrix is a rectangular array of
numbers; a vector is a single ordered list. Superscript $T$ exchanges rows
and columns, and a diagonal matrix has zero entries except on its diagonal.
The product of two matrices forms each entry by summing the paired products
of one row and one column. The identity matrix, written $I$, has ones on its
diagonal and leaves a vector unchanged when multiplied by it.

Let $P=O/n$ contain joint proportions, with positive row and column margins
$r$ and $c$. Write $D_r$ and $D_c$ for diagonal matrices containing those
margins. The transformed table is

$$ S=D_r^{-1/2}(P-rc^T)D_c^{-1/2},
 \qquad S_{ij}=\frac{p_{ij}-r_ic_j}{\sqrt{r_ic_j}}. $$

Here $D_r^{-1/2}$ has entries $1/\sqrt{r_i}$ on its diagonal, and similarly
for columns. Direct substitution gives

$$ \sum_{i,j}S_{ij}^2=\frac{\chi^2}{n}. $$


A singular value decomposition writes $S=U\Sigma V^T$, using unit-length,
mutually perpendicular columns in $U$ and $V$. The positive diagonal entries
$\sigma_1\geq\sigma_2\geq\cdots$ of $\Sigma$ give the sizes of these
components. In symbols, $U^TU=V^TV=I$. This is the numerical decomposition
returned by `numpy.linalg.svd`; the coordinate construction below
uses its defining properties.

The row and column principal coordinates are

$$ F=D_r^{-1/2}U\Sigma,\qquad
 G=D_c^{-1/2}V\Sigma. $$

To verify what a row distance represents, put
$Z=D_r^{-1/2}S$. Its entries are
$Z_{ij}=(q_{ij}-c_j)/\sqrt{c_j}$ and $Z=FV^T$.
The squared distance between two rows of $Z$ is therefore precisely
$\sum_j(q_{ij}-q_{i'j})^2/c_j$.
Because $V^TV=I$, the same squared distance equals

$$ (F_i-F_{i'})V^TV(F_i-F_{i'})^T
 =\sum_k(F_{ik}-F_{i'k})^2. $$

Thus, when all nonzero components are retained, ordinary plotted distances
between row points equal chi-square distances between row profiles.
Interchanging rows and columns proves the corresponding statement for $G$.
There is no comparable identity for the distance between a row point and a
column point.

The contribution of component $k$ to the sum of squared entries of $S$ is
$\sigma_k^2$: expand $S_{ij}=\sum_k\sigma_kU_{ik}V_{jk}$, square, and sum;
all cross-terms vanish because distinct columns are perpendicular.
Hence $\sum_k\sigma_k^2=\chi^2/n$. Keeping the first two components retains
$(\sigma_1^2+\sigma_2^2)/\sum_k\sigma_k^2$ of the inertia.
For any unit direction $v$, its captured sum of squares is
$\|Sv\|^2=\sum_k\sigma_k^2(V_k^Tv)^2\leq\sigma_1^2$,
where $\|\cdot\|^2$ means the sum of squared entries. The weights
$(V_k^Tv)^2$ sum to at most 1; equality at the maximum is attained by
$v=V_1$. Repeating in the perpendicular space gives the second direction.
This establishes the axis ordering used in the map.

Both $S\sqrt{c}=0$ and $\sqrt{r}^{\,T}S=0$, because observed and expected
margins agree. Consequently the number of nonzero components is at most
$\min(R-1,C-1)$ for an $R\times C$ table, and the weighted coordinate means
are zero. The origin is the overall profile. The chapter's $3\times5$
table needs only two components. In a larger table, omitted components
subtract nonnegative squared coordinate differences from a plotted distance;
nearby projected points can conceal differences in omitted dimensions.
Reversing an axis sign preserves every distance and interpretation.

---
*Synthetic data (`generate_data.py`, seed 60). Teaching values only; nothing here is an estimate for a real department.*